# Random Forest na Classificação do Desempenho dos Alunos

# Seção 1: Importação do Dataset e Dataset Usado

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

In [2]:
df = pd.read_parquet("dados_treino.parquet")

# Seção 4: Treinando o Random Forest

In [3]:
def treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf):
    
    rf= RandomForestClassifier(
        n_estimators=n_estimators,        
        max_depth=max_depth,            
        max_features = max_features,     
        min_samples_split = min_samples_split,    
        min_samples_leaf = min_samples_leaf,      
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(x_train, y_train)

    y_pred_train = rf.predict(x_train)
    y_pred_test  = rf.predict(x_test)

    ein  = 1 - accuracy_score(y_train, y_pred_train)
    eout = 1 - accuracy_score(y_test,  y_pred_test)

    print(f"\nEin:  {ein:.4f}")
    print(f"Eout: {eout:.4f}")
    print(f"Gap:  {eout - ein:.4f}  {'overfitting' if eout - ein > 0.05 else 'ok'}")
    print("\n" + classification_report(y_test, y_pred_test))

    return rf

In [4]:
df_reduzido = df.sample(n=3_000_000, random_state=42)

In [5]:
Features = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_FAIXA_ETARIA', 'TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA',
            "IDADE_ALTA", "ACESSO_DIGITAL", 'PAIS_SEM_INSTRUCAO', 'SEM_DIGITAL', 'FAMILIA_GRANDE_POBRE', 'SCORE_BENS_RELEVANTES',
            'TREINEIRO_JOVEM', 'FAMILIA_QUALIFICADA', 'SCORE_RISCO', 'MORA_SOZINHO', 'PUBLICO_CURSANDO', 'SEM_BENS', 'NU_ANO',
            'TP_ST_CONCLUSAO', 'TP_DEPENDENCIA_ADM_ESC', 'TP_SEXO']


X = df_reduzido[Features]
y = df_reduzido['FALTOU']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTreino: {len(x_train):,} | Teste: {len(x_test):,}")
print(f"Taxa de falta no treino: {y_train.mean():.3f}")
print(f"Taxa de falta no teste:  {y_test.mean():.3f}")


Treino: 2,400,000 | Teste: 600,000
Taxa de falta no treino: 0.334
Taxa de falta no teste:  0.334


In [25]:
rf = RandomForestClassifier(random_state=42)

rf.fit(x_train, y_train)

print('Ein: %0.4f' % (1 - accuracy_score(y_train, rf.predict(x_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test, rf.predict(x_test))))
print(classification_report(y_test, rf.predict(x_test)))

Ein: 0.0017
Eout: 0.2918
              precision    recall  f1-score   support

           0       0.74      0.87      0.80     66621
           1       0.60      0.38      0.47     33379

    accuracy                           0.71    100000
   macro avg       0.67      0.63      0.63    100000
weighted avg       0.69      0.71      0.69    100000



# Treinando Usando Parâmetros Atualizados

In [ ]:
n_estimators=100       
max_depth=15            
max_features='log2'    
min_samples_split=20   
min_samples_leaf=50    
    
rf = treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf)

# Verificando a Probabilidade do Aluno ir à Prova

In [ ]:
y_prob = rf.predict_proba(x_test)[:,1]
roc_auc_score(y_test, y_prob)

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_prob
)